In [ ]:
# ---
# jupyter:
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# %% [markdown]
# # GAIA Cassava Quickstart
# Train a TinyViT on cassava leaf disease dataset from Kaggle.

# %% [markdown]
# ### 1. Install dependencies & download data

# %%
!pip install pytorch-lightning torchmetrics kagglehub opencv-python-headless pandas scikit-learn pillow -q

# %%
import os, torch, cv2, pandas as pd, numpy as np
import pytorch_lightning as pl
import kagglehub

# %% [markdown]
# ### 2. Download Cassava Leaf Disease dataset

# %%
path = kagglehub.dataset_download("mexwell/cassava-leaf-disease-classification")
print("Dataset path:", path)

# %% [markdown]
# ### 3. Preprocess: convert to grayscale, resize, create CSV

# %%
RAW_DIR = os.path.join(path, "train")  # adjust if needed
PROC_DIR = "/content/cassava_processed"
os.makedirs(os.path.join(PROC_DIR, "images"), exist_ok=True)
records = []
for class_idx in range(5):
    class_dir = os.path.join(RAW_DIR, str(class_idx))
    if not os.path.exists(class_dir): continue
    for fname in os.listdir(class_dir):
        if not fname.lower().endswith(('.jpg','.jpeg','.png')): continue
        img = cv2.imread(os.path.join(class_dir, fname), cv2.IMREAD_COLOR)
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        resized = cv2.resize(gray, (224, 224))
        cv2.imwrite(os.path.join(PROC_DIR, "images", fname), resized)
        records.append({"image_path": fname, "label": class_idx})
df = pd.DataFrame(records)
df.to_csv(os.path.join(PROC_DIR, "all.csv"), index=False)
print("Preprocessed images:", len(df))

# %% [markdown]
# ### 4. Split train/val

# %%
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df.to_csv(os.path.join(PROC_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(PROC_DIR, "val.csv"), index=False)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

# %% [markdown]
# ### 5. Define TinyViT and Lightning model (same as src)

# %%
# (Copy the exact TinyViT and GAIAModel code here, or import from cloned repo)
# For brevity, we import from our local GAIA library if mounted.
# In Colab, we can just define them inline.
# Provided below for completeness.

# %%
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import torchvision.transforms as T

class TransformerBlock(nn.Module):
    # ... (same code) ...

class TinyViT(nn.Module):
    # ... (same code) ...

class GAIAModel(pl.LightningModule):
    # ... (same code) ...

# %%
# Dataset
class CassavaDataset(Dataset):
    def __init__(self, csv_path, img_dir, augment=False):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        if augment:
            self.transform = T.Compose([
                T.RandomResizedCrop(224, scale=(0.8, 1.0)),
                T.RandomHorizontalFlip(0.5),
                T.RandomRotation(15),
                T.ColorJitter(0.2, 0.2),
                T.ToTensor(),
            ])
        else:
            self.transform = T.Compose([T.Resize((224,224)), T.ToTensor()])
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['image_path'])).convert('L')
        img = self.transform(img)
        return img, int(row['label'])

# %%
train_ds = CassavaDataset(os.path.join(PROC_DIR, "train.csv"), os.path.join(PROC_DIR, "images"), augment=True)
val_ds = CassavaDataset(os.path.join(PROC_DIR, "val.csv"), os.path.join(PROC_DIR, "images"), augment=False)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, num_workers=2)

# %%
model = GAIAModel(num_classes=5, lr=1e-3)
trainer = pl.Trainer(max_epochs=20, accelerator='gpu', devices=1,
                     callbacks=[pl.callbacks.EarlyStopping(monitor='val_loss', patience=5)])
trainer.fit(model, train_loader, val_loader)

# %%
torch.save(model.state_dict(), "gaia_cassava.pt")
print("Model saved.")